# Phase 4: DPO Training Report

## 1. What DPO Does and Why

The LLM implicitly learns a reward function by maximizing the log-probability of **Chosen** responses and minimizing it for **Rejected** responses. Instead of using RLHF, no separate reward model is needed. No unstable PPO loop. DPO transforms alignment into a binary classification problem, leveraging the LLM's own implicit reward structure — much faster and more stable training.

This phase takes the preference pairs from Phase 3 (`data/preferences/dpo_dataset.jsonl`) and trains the SFT checkpoint to prefer the chosen code over the rejected code. The key parameter is **β (beta)**: it controls how far the aligned model is allowed to deviate from the SFT reference policy — too low and the model barely changes, too high and it overfits to the preference data.

Dependencies (`pandas`, `matplotlib`, `seaborn`) are in the `dev` dependency group in `pyproject.toml` — run `uv sync` rather than `pip install`ing them inline.

## 2. Hyperparameters

Identical LoRA / QLoRA configuration to SFT (golden recipe) — the only new parameter is `beta`, which is DPO-specific.

| Parameter | Value | Notes |
|---|---|---|
| `BETA` | `0.1` | Standard DPO β — controls KL divergence from the SFT reference policy |
| `LORA_R` | `16` | Same as SFT |
| `LORA_ALPHA` | `32` | Same as SFT |
| `LORA_DROPOUT` | `0.1` | Same as SFT |
| `LEARNING_RATE` | `2e-4` | Same as SFT |
| `PER_DEVICE_BATCH_SIZE` | `2` | Same as SFT |
| `GRAD_ACCUMULATION_STEPS` | `8` | Effective batch size 16 |
| `NUM_TRAIN_EPOCHS` | `3` | Starting point — check loss curve |
| `MAX_SEQ_LENGTH` | `2048` | Same as SFT |
| `OPTIMIZER` | `paged_adamw_8bit` | Same as SFT |
| Quantization | NF4 + double quant + bfloat16 | QLoRA — same as SFT |

**Why identical LoRA config?** The golden-recipe rationale from Phase 2 applies here too — the compute budget is better spent on the evaluation ablation (Phase 5) than on DPO hyperparameter sweeps. The one parameter that *is* DPO-specific (`beta=0.1`) uses the standard value from the [DPO paper](https://arxiv.org/abs/2305.18290).

## 3. Lab Journal: Training Observations

### Weights & Biases logging

- **W&B run name:** `dpo-qwen2.5-coder-7b-composite-reward`
- **W&B report link:** [add here]

### DPO Loss

The DPO loss is:

$$\mathcal{L}_{\text{DPO}} = -\log\sigma\left(\beta \left[ \log\frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \log\frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)} \right]\right)$$

Where $y_w$ = chosen, $y_l$ = rejected, $\pi_\theta$ = current policy, $\pi_{\text{ref}}$ = frozen SFT policy.

**What to look for:**
- Loss should decrease smoothly — if it's noisy, the preference pairs may be low quality
- If loss plateaus quickly, β may be too high (model can't deviate enough from reference)
- `rewards/chosen` should increase, `rewards/rejected` should decrease — this is the margin DPO is learning

### Observations

*(Fill in after the run:)*
- **Training time:** [hours, GPU model]
- **VRAM usage:** [expected ~16–18 GB, same as SFT]
- **Loss convergence:** [smooth? noisy? plateaued early?]
- **rewards/margins:** [did chosen-rejected gap widen?]

## 4. Evaluation: Base vs. SFT vs. DPO

The three-way comparison that answers the project's core question. Same `bigcode-evaluation-harness` setup as notebook 02 — the DPO cell below is the new addition.

### 4a. DPO checkpoint — SFT model + DPO adapter

In [ ]:
!accelerate launch tools/bigcode-evaluation-harness/main.py \
  --model checkpoints/sft/merged_model \
  --peft_model checkpoints/dpo/final_model \
  --load_in_4bit \
  --tasks humaneval \
  --precision bf16 \
  --allow_code_execution \
  --save_generations \
  --save_generations_path data/evaluation/dpo_generations.json \
  --metric_output_path data/evaluation/dpo_metrics.json \
  --limit 20

### 4b. Three-way comparison

In [ ]:
import json
import pandas as pd

def load_metrics(filepath):
    try:
        with open(filepath, "r") as f:
            return json.load(f)
    except FileNotFoundError:
        return None

base_m = load_metrics("data/evaluation/baseline_metrics.json")
sft_m  = load_metrics("data/evaluation/sft_metrics.json")
dpo_m  = load_metrics("data/evaluation/dpo_metrics.json")

rows = []
for label, m in [("Base", base_m), ("SFT", sft_m), ("DPO (composite)", dpo_m)]:
    if m:
        rows.append({"Model": label, "pass@1 (%)": m.get("humaneval", {}).get("pass@1", 0) * 100})
    else:
        rows.append({"Model": label, "pass@1 (%)": "pending"})

df = pd.DataFrame(rows)
print(df.to_string(index=False))

### 4c. Visual comparison

In [ ]:
import matplotlib.pyplot as plt

# Only plot if all metrics are available
if all([base_m, sft_m, dpo_m]):
    models = ["Base", "SFT", "DPO\n(composite)"]
    scores = [
        base_m["humaneval"]["pass@1"] * 100,
        sft_m["humaneval"]["pass@1"] * 100,
        dpo_m["humaneval"]["pass@1"] * 100,
    ]
    colors = ["#9E9E9E", "#2196F3", "#4CAF50"]

    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(models, scores, color=colors, width=0.5)
    ax.set_ylabel("HumanEval pass@1 (%)")
    ax.set_title("Base vs. SFT vs. DPO (Composite Reward)")
    ax.set_ylim(0, 100)
    for bar, score in zip(bars, scores):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                f"{score:.1f}%", ha="center", fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print("Not all metrics available yet — run evaluation cells first.")

## 5. Key Takeaways

*(Fill in after running Phase 4 and evaluation:)*

- **DPO vs. SFT:** [did pass@1 improve? by how much?]
- **Training stability:** [was the DPO loss smooth? any signs of overfitting?]
- **Composite reward effect:** [this is answered more fully in Phase 5's ablation — but initial impressions from the loss curves?]
- **Next step:** Phase 5 — full evaluation with ablation (DPO composite vs. DPO execution-only) to answer the core research question.